In [ ]:
"""
Version Manager — Saves EVERY run (pass or fail) with full audit trail.
========================================================================
Nothing gets deleted. Every attempt is timestamped and checksummed.
"""

In [ ]:
import json
import hashlib
import os
import platform
from datetime import datetime, timezone

In [ ]:
VERSION_STORE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "version_store")

In [ ]:
def get_environment():
    """Capture full environment fingerprint."""
    env = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    try:
        import jax
        import numpy as np
        env["jax_version"] = jax.__version__
        env["numpy_version"] = np.__version__
        env["jax_devices"] = str(jax.devices())
        env["jax_backend"] = jax.default_backend()
    except ImportError:
        env["jax_version"] = "NOT INSTALLED"
    return env

In [ ]:
def compute_checksum(data):
    """SHA-256 checksum of serialized data."""
    data_str = json.dumps(data, sort_keys=True, default=str)
    return hashlib.sha256(data_str.encode()).hexdigest()

In [ ]:
def save_run(run_name, config_snapshot, results, status="unknown"):
    """
    Save a complete run to the version store.

    Args:
        run_name: Short name for this run (e.g., "test_1a_propagation")
        config_snapshot: Dict of all parameters used
        results: Dict of test results
        status: "pass", "fail", "partial", "error"
    """
    os.makedirs(VERSION_STORE, exist_ok=True)

    ts = datetime.now(timezone.utc)
    run_id = ts.strftime("%Y%m%d_%H%M%S") + f"_{run_name}"

    record = {
        "run_id": run_id,
        "run_name": run_name,
        "status": status,
        "timestamp_utc": ts.isoformat(),
        "environment": get_environment(),
        "config": config_snapshot,
        "results": results,
    }

    record["checksum"] = compute_checksum(record)

    filepath = os.path.join(VERSION_STORE, f"{run_id}.json")
    with open(filepath, "w") as f:
        json.dump(record, f, indent=2, default=str)

    # Also append to the run index
    index_path = os.path.join(VERSION_STORE, "RUN_INDEX.txt")
    with open(index_path, "a") as f:
        f.write(f"{ts.isoformat()} | {run_name} | {status} | {record['checksum'][:16]}\n")

    print(f"Run saved: {filepath}")
    print(f"Checksum: {record['checksum'][:16]}")
    return record

In [ ]:
def list_runs():
    """List all saved runs."""
    index_path = os.path.join(VERSION_STORE, "RUN_INDEX.txt")
    if os.path.exists(index_path):
        with open(index_path, "r") as f:
            return f.read()
    return "No runs recorded yet."